In [1]:
!pip install duckdb

In [2]:
import duckdb

from google.colab import drive
drive.mount('/content/drive')

con = duckdb.connect()

print("Ambiente configurado e banco conectado com sucesso!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ambiente configurado e banco conectado com sucesso!


# **1. Ingestão de Dados (Camada Raw / Bronze)**




In [3]:
caminho_base = '/content/drive/MyDrive/case_itau/home-credit-default-risk/'

arquivos_csv = [
    'application_train',
    'application_test',
    'bureau',
    'bureau_balance',
    'previous_application',
    'POS_CASH_balance',
    'credit_card_balance',
    'installments_payments'
]

# Loop iterando sobre a lista para criar todas as views da Camada Raw dinamicamente
for nome_arquivo in arquivos_csv:
    caminho_completo = f"{caminho_base}{nome_arquivo}.csv"

    query = f"""
        CREATE OR REPLACE VIEW raw_{nome_arquivo} AS
        SELECT * FROM read_csv_auto('{caminho_completo}');
    """
    try:
        con.execute(query)
        print(f"ok - View raw_{nome_arquivo} criada com sucesso!")
    except Exception as e:
        print(f"x - Erro ao criar view raw_{nome_arquivo}. Verifique se o arquivo existe neste caminho. Erro: {e}")

print("\n--- Tabelas e Views no Banco ---")
display(con.execute("SHOW TABLES;").df())

ok - View raw_application_train criada com sucesso!
ok - View raw_application_test criada com sucesso!
ok - View raw_bureau criada com sucesso!
ok - View raw_bureau_balance criada com sucesso!
ok - View raw_previous_application criada com sucesso!
ok - View raw_POS_CASH_balance criada com sucesso!
ok - View raw_credit_card_balance criada com sucesso!
ok - View raw_installments_payments criada com sucesso!

--- Tabelas e Views no Banco ---


,name
0,raw_POS_CASH_balance
1,raw_application_test
2,raw_application_train
3,raw_bureau
4,raw_bureau_balance
5,raw_credit_card_balance
6,raw_installments_payments
7,raw_previous_application


# **2. Limpeza e Padronização (Camada Staging / Silver)**

In [4]:
query_qualidade = """
SELECT
    COUNT(*) AS total_linhas,
    COUNT(DISTINCT SK_ID_CURR) AS ids_unicos,

    SUM(CASE WHEN AMT_CREDIT IS NULL THEN 1 ELSE 0 END) AS nulos_credito,
    SUM(CASE WHEN AMT_ANNUITY IS NULL THEN 1 ELSE 0 END) AS nulos_anuidade,
    SUM(CASE WHEN AMT_GOODS_PRICE IS NULL THEN 1 ELSE 0 END) AS nulos_preco_bem,

    SUM(CASE WHEN DAYS_EMPLOYED = 365243 THEN 1 ELSE 0 END) AS anomalias_dias_trabalhados

FROM raw_application_train;
"""

display(con.execute(query_qualidade).df())

,total_linhas,ids_unicos,nulos_credito,nulos_anuidade,nulos_preco_bem,anomalias_dias_trabalhados
0,307511,307511,0.0,12.0,278.0,55374.0


In [5]:
query_staging = """
CREATE OR REPLACE VIEW stg_application_train AS
WITH stats AS (
    SELECT
        MEDIAN(AMT_ANNUITY) AS med_annuity,
        MEDIAN(AMT_GOODS_PRICE) AS med_goods
    FROM raw_application_train
)
SELECT
    a.SK_ID_CURR,
    a.TARGET,
    a.NAME_CONTRACT_TYPE,
    a.CODE_GENDER,
    a.FLAG_OWN_CAR,
    a.FLAG_OWN_REALTY,
    a.CNT_CHILDREN,
    a.AMT_INCOME_TOTAL,
    a.AMT_CREDIT,
    a.NAME_INCOME_TYPE,
    a.NAME_EDUCATION_TYPE,
    a.NAME_FAMILY_STATUS,

    -- 1. Tratamento de Nulos (Imputando a mediana)
    COALESCE(a.AMT_ANNUITY, s.med_annuity) AS AMT_ANNUITY,
    COALESCE(a.AMT_GOODS_PRICE, s.med_goods) AS AMT_GOODS_PRICE,

    -- 2. Tratamento da anomalia de dias trabalhados
    CASE
        WHEN a.DAYS_EMPLOYED = 365243 THEN NULL
        ELSE ABS(a.DAYS_EMPLOYED) / 365.0
    END AS YEARS_EMPLOYED,

    -- 3. Features Derivadas / Enriquecimento
    ABS(a.DAYS_BIRTH) / 365.0 AS AGE_YEARS,
    a.AMT_CREDIT / NULLIF(a.AMT_INCOME_TOTAL, 0) AS CREDIT_INCOME_RATIO,
    COALESCE(a.AMT_ANNUITY, s.med_annuity) / NULLIF(a.AMT_INCOME_TOTAL, 0) AS ANNUITY_INCOME_RATIO

FROM raw_application_train a
CROSS JOIN stats s;
"""

con.execute(query_staging)
print("View stg_application_train criada com sucesso!")

display(con.execute("SELECT SK_ID_CURR, AGE_YEARS, YEARS_EMPLOYED, CREDIT_INCOME_RATIO FROM stg_application_train LIMIT 5").df())

View stg_application_train criada com sucesso!


,SK_ID_CURR,AGE_YEARS,YEARS_EMPLOYED,CREDIT_INCOME_RATIO
0,100002,25.920548,1.745205,2.007889
1,100003,45.931507,3.254795,4.790750
2,100004,52.180822,0.616438,2.000000
3,100006,52.068493,8.326027,2.316167
4,100007,54.608219,8.323288,4.222222


# **3. Agregações e Granularidade (Camada Intermediate / Silver)**

In [6]:
query_bureau = """
CREATE OR REPLACE VIEW int_bureau_summary AS
SELECT
    SK_ID_CURR,
    COUNT(SK_ID_BUREAU) AS BUREAU_TOTAL_LOANS,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN 1 ELSE 0 END) AS BUREAU_ACTIVE_LOANS,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Closed' THEN 1 ELSE 0 END) AS BUREAU_CLOSED_LOANS,

    SUM(AMT_CREDIT_SUM) AS BUREAU_TOTAL_CREDIT_LIMIT,
    SUM(AMT_CREDIT_SUM_DEBT) AS BUREAU_TOTAL_DEBT,

    AVG(CREDIT_DAY_OVERDUE) AS BUREAU_AVG_OVERDUE_DAYS
FROM raw_bureau
GROUP BY SK_ID_CURR;
"""

con.execute(query_bureau)
print("View int_bureau_summary criada com sucesso!")

display(con.execute("SELECT * FROM int_bureau_summary LIMIT 5").df())

View int_bureau_summary criada com sucesso!


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,SK_ID_CURR,BUREAU_TOTAL_LOANS,BUREAU_ACTIVE_LOANS,BUREAU_CLOSED_LOANS,BUREAU_TOTAL_CREDIT_LIMIT,BUREAU_TOTAL_DEBT,BUREAU_AVG_OVERDUE_DAYS
0,215354,11,6.0,5.0,5973945.30,284463.18,0.0
1,162297,6,3.0,3.0,8230386.15,0.00,0.0
2,402440,1,1.0,0.0,89910.00,76905.00,0.0
3,238881,8,3.0,5.0,1285239.06,552730.50,0.0
4,222183,8,5.0,3.0,7158960.00,1185081.84,0.0


In [7]:
query_previous = """
CREATE OR REPLACE VIEW int_previous_application_summary AS
SELECT
    SK_ID_CURR,
    COUNT(SK_ID_PREV) AS PREV_TOTAL_APPLICATIONS,

    SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 ELSE 0 END) AS PREV_APPROVED_COUNT,
    SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Refused' THEN 1 ELSE 0 END) AS PREV_REFUSED_COUNT,

    SUM(AMT_APPLICATION) AS PREV_TOTAL_AMT_REQUESTED,
    SUM(AMT_CREDIT) AS PREV_TOTAL_AMT_APPROVED
FROM raw_previous_application
GROUP BY SK_ID_CURR;
"""

con.execute(query_previous)
print("View int_previous_application_summary criada com sucesso!")

display(con.execute("SELECT * FROM int_previous_application_summary LIMIT 5").df())

View int_previous_application_summary criada com sucesso!


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,SK_ID_CURR,PREV_TOTAL_APPLICATIONS,PREV_APPROVED_COUNT,PREV_REFUSED_COUNT,PREV_TOTAL_AMT_REQUESTED,PREV_TOTAL_AMT_APPROVED
0,284098,22,10.0,7.0,6629831.55,7356681.0
1,105378,10,7.0,1.0,2530345.50,2916909.0
2,306381,34,9.0,18.0,17860995.00,19617646.5
3,152109,7,7.0,0.0,1462141.44,1824813.0
4,336375,11,5.0,5.0,4145212.17,4959513.0


# **4. Camada Analítica Final (Camada OBT / Gold)**

In [8]:
query_obt = """
CREATE OR REPLACE TABLE obt_credit_risk_features AS
SELECT
    a.*,

    COALESCE(b.BUREAU_TOTAL_LOANS, 0) AS BUREAU_TOTAL_LOANS,
    COALESCE(b.BUREAU_ACTIVE_LOANS, 0) AS BUREAU_ACTIVE_LOANS,
    COALESCE(b.BUREAU_CLOSED_LOANS, 0) AS BUREAU_CLOSED_LOANS,
    COALESCE(b.BUREAU_TOTAL_CREDIT_LIMIT, 0) AS BUREAU_TOTAL_CREDIT_LIMIT,
    COALESCE(b.BUREAU_TOTAL_DEBT, 0) AS BUREAU_TOTAL_DEBT,
    COALESCE(b.BUREAU_AVG_OVERDUE_DAYS, 0) AS BUREAU_AVG_OVERDUE_DAYS,

    COALESCE(p.PREV_TOTAL_APPLICATIONS, 0) AS PREV_TOTAL_APPLICATIONS,
    COALESCE(p.PREV_APPROVED_COUNT, 0) AS PREV_APPROVED_COUNT,
    COALESCE(p.PREV_REFUSED_COUNT, 0) AS PREV_REFUSED_COUNT,
    COALESCE(p.PREV_TOTAL_AMT_REQUESTED, 0) AS PREV_TOTAL_AMT_REQUESTED,
    COALESCE(p.PREV_TOTAL_AMT_APPROVED, 0) AS PREV_TOTAL_AMT_APPROVED

FROM stg_application_train a
LEFT JOIN int_bureau_summary b
    ON a.SK_ID_CURR = b.SK_ID_CURR
LEFT JOIN int_previous_application_summary p
    ON a.SK_ID_CURR = p.SK_ID_CURR;
"""

con.execute(query_obt)
print("Tabela Analítica (OBT) criada com sucesso!")

display(con.execute("SELECT COUNT(*), COUNT(DISTINCT SK_ID_CURR) FROM obt_credit_risk_features;").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Tabela Analítica (OBT) criada com sucesso!


,count_star(),count(DISTINCT SK_ID_CURR)
0,307511,307511


# **5. Análises Exploratórias**

In [9]:
query_metricas_renda = """
SELECT
    NAME_INCOME_TYPE AS tipo_renda,
    COUNT(*) AS total_clientes,
    ROUND(AVG(TARGET) * 100, 2) AS taxa_inadimplencia_pct,
    ROUND(AVG(AGE_YEARS), 1) AS idade_media,
    ROUND(AVG(AMT_CREDIT), 2) AS ticket_medio_credito
FROM obt_credit_risk_features
GROUP BY NAME_INCOME_TYPE
ORDER BY total_clientes DESC;
"""

print("--- Perfil de Risco por Tipo de Renda ---")
display(con.execute(query_metricas_renda).df())

--- Perfil de Risco por Tipo de Renda ---


,tipo_renda,total_clientes,taxa_inadimplencia_pct,idade_media,ticket_medio_credito
0,Working,158774,9.59,40.4,577011.03
1,Commercial associate,71617,7.48,40.3,669913.12
2,Pensioner,55362,5.39,59.8,542546.06
3,State servant,21703,5.75,41.3,669819.30
4,Unemployed,22,36.36,44.4,764386.36
5,Student,18,0.00,39.2,510787.50
6,Businessman,10,0.00,44.0,1228500.00
7,Maternity leave,5,40.00,40.6,749700.00


In [10]:
query_recusas = """
SELECT
    CASE
        WHEN PREV_REFUSED_COUNT = 0 THEN '1. Nenhuma recusa prévia'
        WHEN PREV_REFUSED_COUNT BETWEEN 1 AND 2 THEN '2. 1 a 2 recusas'
        ELSE '3. 3 ou mais recusas'
    END AS historico_recusas,
    COUNT(*) AS total_clientes,
    ROUND(AVG(TARGET) * 100, 2) AS taxa_inadimplencia_pct
FROM obt_credit_risk_features
GROUP BY 1
ORDER BY 1;
"""

print("--- Impacto de Recusas Anteriores na Inadimplência ---")
display(con.execute(query_recusas).df())

--- Impacto de Recusas Anteriores na Inadimplência ---


,historico_recusas,total_clientes,taxa_inadimplencia_pct
0,1. Nenhuma recusa prévia,207217,6.98
1,2. 1 a 2 recusas,69273,9.30
2,3. 3 ou mais recusas,31021,12.61


In [11]:
query_comprometimento = """
SELECT
    CASE
        WHEN ANNUITY_INCOME_RATIO < 0.10 THEN '1. Baixo (< 10% da renda)'
        WHEN ANNUITY_INCOME_RATIO >= 0.10 AND ANNUITY_INCOME_RATIO < 0.20 THEN '2. Médio (10% a 20%)'
        WHEN ANNUITY_INCOME_RATIO >= 0.20 AND ANNUITY_INCOME_RATIO < 0.30 THEN '3. Alto (20% a 30%)'
        ELSE '4. Muito Alto (> 30% da renda)'
    END AS faixa_comprometimento,
    COUNT(*) AS total_clientes,
    ROUND(AVG(TARGET) * 100, 2) AS taxa_inadimplencia_pct
FROM obt_credit_risk_features
WHERE ANNUITY_INCOME_RATIO IS NOT NULL
GROUP BY 1
ORDER BY 1;
"""

print("\n--- Impacto do Comprometimento de Renda ---")
display(con.execute(query_comprometimento).df())


--- Impacto do Comprometimento de Renda ---


,faixa_comprometimento,total_clientes,taxa_inadimplencia_pct
0,1. Baixo (< 10% da renda),54270,7.26
1,2. Médio (10% a 20%),147875,8.01
2,3. Alto (20% a 30%),73938,8.76
3,4. Muito Alto (> 30% da renda),31428,8.15


# **6. Exportação**

In [12]:
caminho_exportacao = '/content/drive/MyDrive/case_itau/obt_credit_risk_features.parquet'

con.execute(f"COPY obt_credit_risk_features TO '{caminho_exportacao}' (FORMAT PARQUET);")

print(f"Tabela Analítica exportada com sucesso para: {caminho_exportacao}")

Tabela Analítica exportada com sucesso para: /content/drive/MyDrive/case_itau/obt_credit_risk_features.parquet


In [13]:
caminho_csv = '/content/drive/MyDrive/case_itau/obt_credit_risk_features.csv'
con.execute(f"COPY obt_credit_risk_features TO '{caminho_csv}' (FORMAT CSV, HEADER);")
print(f"CSV gerado com sucesso para: {caminho_csv}")

CSV gerado com sucesso para: /content/drive/MyDrive/case_itau/obt_credit_risk_features.csv
